# BellaBox Product CNN من ملف Excel

هذا الدفتر يبني Dataset من **تصدير منتجات سلة Excel** الذي يحتوي على اسم المنتج، تصنيف المنتج، وعمود `صورة المنتج`. لا يستخدم Sitemap ولا Dataset تعليميًا جاهزًا.

فعّل GPU من `Runtime > Change runtime type > T4 GPU` واربط Google Drive لحفظ الصور وCheckpoints بعد انتهاء جلسة Colab.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path('/content/bellabox-product-cnn')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/mohammedalhmed/bellabox-product-cnn.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'ml/requirements-colab.txt')], check=True)
print('Repository ready:', REPO_DIR)

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

WORK_DIR = Path('/content/drive/MyDrive/BellaBox_Product_CNN')
DATA_DIR = WORK_DIR / 'dataset'
OUTPUT_DIR = WORK_DIR / 'outputs'
XLSX_PATH = WORK_DIR / 'bellabox_products.xlsx'
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not XLSX_PATH.exists():
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    shutil.copy2(uploaded_name, XLSX_PATH)
print('Excel source:', XLSX_PATH)
print('Persistent work directory:', WORK_DIR)

## 1) بناء Dataset من ملف Excel

الأداة تقرأ صف العناوين الثاني في قالب سلة، وتستخدم `تصنيف المنتج` كـ label و`صورة المنتج` كمصدر للصور. المستوى 2 يعني مثلًا: `العناية > العناية بالوجه`. يتم استبعاد الفئات الصغيرة تلقائيًا وتسجيلها في `dataset_summary.json`. راجع `manifest.csv` قبل التدريب.

In [ ]:
build_cmd = [
    sys.executable, str(REPO_DIR / 'ml/build_dataset.py'),
    '--products-xlsx', str(XLSX_PATH),
    '--output-dir', str(DATA_DIR),
    '--category-level', '2',
    '--min-images-per-class', '20',
    '--min-products-per-class', '4',
    '--drop-small-classes',
    '--max-images-per-product', '3',
    '--download',
]
subprocess.run(build_cmd, check=True)
summary = json.loads((DATA_DIR / 'dataset_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 2) تدريب CNN مع Checkpoints

التدريب ينفذ Grouped Split حسب `product_id`، ويستخدم EfficientNetB0، Augmentation، Class Weights، Early Stopping، ReduceLROnPlateau، و`BackupAndRestore`. يمكن إعادة تشغيل الخلية بعد انقطاع Colab؛ سيقرأ ملفات الاستئناف الموجودة في Drive.

In [ ]:
train_cmd = [
    sys.executable, str(REPO_DIR / 'ml/train.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '15',
    '--batch-size', '32',
    '--resume',
]
subprocess.run(train_cmd, check=True)
print(json.dumps(json.loads((OUTPUT_DIR / 'metrics.json').read_text()), indent=2))

## 3) اختبار النموذج على صورة جديدة

ارفع صورة منتج من خارج Dataset إن أمكن، ثم راقب Top-3. إذا كانت الثقة منخفضة أو الصورة مختلفة عن صور التدريب، تُحال النتيجة للمراجعة البشرية.

In [ ]:
uploaded_test = files.upload()
test_image = next(iter(uploaded_test))
predict_cmd = [
    sys.executable, str(REPO_DIR / 'ml/predict.py'),
    '--model', str(OUTPUT_DIR / 'final_model.keras'),
    '--labels', str(OUTPUT_DIR / 'labels.json'),
    '--image', test_image,
    '--top-k', '3',
]
subprocess.run(predict_cmd, check=True)